##Parte 1: Início
definição das importações e início do carregamento dos arquivos do dataset e exibição de imagens do lote para visualização simples


In [1]:
## definição das importações
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt
import numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
print("SETUP E CARREGAMENTO DE DADOS (SVHN)")

# 1. Definição das transformações iniciais
# Converte a img PIL para tensor e normaliza os valores de pixels para -1 e 1
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 2. Carregamento do dataset SVHN (PyTorch baixa com base nos links oficiais)
print("\n [1/2] Baixando e carregando dados de TREINO (Aprox. 182 MB)...")
train_dataset = torchvision.datasets.SVHN(root='./data', split='train', download=True, transform=transform)

print(" [2/2] Baixando e carregando dados de TESTE (Aprox. 64 MB)...")
test_dataset = torchvision.datasets.SVHN(root='./data', split='test', download=True, transform=transform)

# 3. Criação dos Dataloaders
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("-----CARREGAMENTO CONCLUÍDO-----")
print(f"Total de imagens para Treino : {len(train_dataset):,}".replace(',', '.'))
print(f"Total de imagens para Teste  : {len(test_dataset):,}".replace(',', '.'))


In [ ]:
def imshow(img):
    """Função para desnormalizar e plotar um tensor de imagem na tela."""
    # Desnormalização: Reverte a operação ((x - 0.5) / 0.5)
    img = img / 2 + 0.5
    npimg = img.numpy()

    plt.figure(figsize=(12, 4))

    # O PyTorch usa o formato (Canais, Altura, Largura) [C, H, W]
    # O Matplotlib exige o formato (Altura, Largura, Canais) [H, W, C]
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')
    plt.title("Amostra de 8 imagens do lote de treino", fontsize=14, fontweight='bold', pad=15)
    plt.show()

# Pegando um único lote de imagens do iterador de treino
dataiter = iter(train_loader)
images, labels = next(dataiter)

print("VISUALIZAÇÃO E RÓTULOS ---> GABARITO")

# Exibição das primeiras 8 imagens do lote
imshow(torchvision.utils.make_grid(images[:8]))

# Formatação limpa dos rótulos
rotulos_str = ' | '.join(f'Dígito {labels[j].item()}' for j in range(8))
print(f"RÓTULOS REAIS: [ {rotulos_str} ]")

#Parte 2: arquietura do baseline cnn


In [ ]:
class BaselineCNN(nn.Module):
    def __init__(self):
        super(BaselineCNN, self).__init__()
        # Imagens SVHN: 3 canais (RGB) de 32x32 pixels
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)

        # Após 2 pools (redução pela metade 2x), a imagem 32x32 vira 8x8
        # 32 canais * 8 * 8 = 2048 características mapeadas
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10) # 10 classes finais (dígitos de 0 a 9)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 8 * 8) # Achatamento (Flatten) para as camadas lineares
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

print("CONFIGURAÇÃO DE HARDWARE E TREINAMENTO BASELINE")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de Treinamento ativo: >>> {device.type.upper()} <<<")

model = BaselineCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

best_acc_baseline = 0.0
caminho_baseline = 'modelo_baseline.pth'
epochs = 5

print(f"\n Iniciando treinamento da rede Baseline por {epochs} épocas...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for i, data in enumerate(train_loader, 0):
        inputs, labeis = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labeis)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    loss_epoca = running_loss / len(train_loader)

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_acc = (correct / total) * 100

    print(f"ÉPOCA [{epoch+1}/{epochs}] | Perda (Loss): {loss_epoca:.4f} | Acurácia: {epoch_acc:.2f}%")

    if epoch_acc > best_acc_baseline:
        best_acc_baseline = epoch_acc
        torch.save(model.state_dict(), caminho_baseline)
        print(f"   ↪🚩 Checkpoint guardado ({best_acc_baseline:.2f}%)")

print(f"Treino da Baseline Concluído ---> O melhor modelo foi salvo em: '{caminho_baseline}'")

#BASELINECNN COM BATCH NORMALIZATION

In [ ]:
class BaselineBatchNormCNN(nn.Module):
    def __init__(self):
        super(BaselineBatchNormCNN, self).__init__()

        # Camada 1: Convolução + Filtro Batch Normalization
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=5, padding=2)
        self.bn1 = nn.BatchNorm2d(16) # <--- FILTRO MATEMÁTICO ADICIONADO
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Camada 2: Segunda Convolução + Filtro Batch Normalization
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=5, padding=2)
        self.bn2 = nn.BatchNorm2d(32) # <--- FILTRO MATEMÁTICO ADICIONADO

        # Camadas Lineares de classificação (Igual à Baseline anterior)
        # Imagem 32x32 -> Pool 1 (16x16) -> Pool 2 (8x8)
        self.fc1 = nn.Linear(32 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        # Fluxo de dados aplicando a normalização ANTES da ativação ReLU
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        # Achatamento (Flatten)
        x = x.view(-1, 32 * 8 * 8)

        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

print("CONFIGURAÇÃO E TREINO: BASELINE + BATCH NORMALIZATION")

# 1. Configurar dispositivo na GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de Treinamento ativo: >>> {device.type.upper()} <<<")

model_bn_test = BaselineBatchNormCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_bn_test.parameters(), lr=0.001)

best_acc_bn = 0.0
caminho_bn = 'modelo_baseline_bn.pth'
epochs = 5

print(f"\nIniciando treinamento Batch Norm por {epochs} épocas...")

for epoch in range(epochs):
    model_bn_test.train()
    running_loss = 0.0

    for i, data in enumerate(train_loader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = model_bn_test(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    loss_epoca = running_loss / len(train_loader)

    model_bn_test.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = model_bn_test(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = (correct / total) * 100

    print(f"ÉPOCA [{epoch+1}/{epochs}] | Perda (Loss): {loss_epoca:.4f} | Acurácia: {epoch_acc:.2f}%")

    if epoch_acc > best_acc_bn:
        best_acc_bn = epoch_acc
        torch.save(model_bn_test.state_dict(), caminho_bn)
        print(f"  ↪🚩 Checkpoint guardado ({best_acc_bn:.2f}%)")

print(f"Treino com Batch Normalization Concluído ---> Melhor modelo salvo em: '{caminho_bn}'")

#PARTE 3: definição da arquitetura profunda (DEEPCNN)


In [ ]:
class DeepCNN(nn.Module):
    def __init__(self):
        super(DeepCNN, self).__init__()

        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop1 = nn.Dropout(0.25) # Desliga 25% dos neurônios para evitar overfitting

        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)
        self.drop2 = nn.Dropout(0.25)

        # A imagem 32x32 passa por dois pools (reduz para 16x16, depois 8x8) com 64 canais
        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.bn5 = nn.BatchNorm1d(512)
        self.drop3 = nn.Dropout(0.5) # Desliga 50% dos neurônios na camada densa
        self.fc2 = nn.Linear(512, 10) # 10 saídas finais (Dígitos 0-9)

    def forward(self, x):
        # Fluxo do Bloco 1
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.pool1(x)
        x = self.drop1(x)

        # Fluxo do Bloco 2
        x = F.relu(self.bn3(self.conv3(x)))
        x = F.relu(self.bn4(self.conv4(x)))
        x = self.pool2(x)
        x = self.drop2(x)

        # Achatamento (Flatten) para entrar nas camadas lineares
        x = x.view(-1, 64 * 8 * 8)

        # Fluxo do Bloco 3
        x = F.relu(self.bn5(self.fc1(x)))
        x = self.drop3(x)
        x = self.fc2(x)
        return x

print("CONFIGURAÇÃO E TREINO ---> REDE PROFUNDA SUPERVISIONADA (DeepCNN)")

# 1. Configurando para usar a GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de Treinamento ativo ---> {device.type.upper()} <<<")

# 2. Instanciamento do modelo
model_deep = DeepCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_deep.parameters(), lr=0.001)

# 3. Configuração do Checkpoint
best_acc_deep = 0.0
caminho_deep = 'modelo_deepcnn.pth'
epochs_deep = 10

print(f"\nIniciando treinamento da DeepCNN por {epochs_deep} épocas...")

for epoch in range(epochs_deep):
    model_deep.train()
    running_loss = 0.0

    for i, data in enumerate(train_loader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer.zero_grad()
        outputs = model_deep(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    loss_epoca = running_loss / len(train_loader)

    model_deep.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = model_deep(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = (correct / total) * 100

    # Output embelezado da época
    print(f"ÉPOCA [{epoch+1}/{epochs_deep}] | Perda (Loss): {loss_epoca:.4f} | Acurácia: {epoch_acc:.2f}%")

    if epoch_acc > best_acc_deep:
        best_acc_deep = epoch_acc
        torch.save(model_deep.state_dict(), caminho_deep)
        print(f"   ↪🚩 Checkpoint guardado ({best_acc_deep:.2f}%)")

print(f"Treino da DeepCNN Concluído ---> melhor modelo salvo em: '{caminho_deep}'")

#PARTE 4: INSERÇÃO DO RANDAUGMENT


In [ ]:
# 1. Definição das transformações com RandAugment
# O RandAugment escolhe aleatoriamente 'num_ops' operações com força 'magnitude'
transform_augment = transforms.Compose([
    transforms.RandAugment(num_ops=2, magnitude=5),
    # Magnitude de 5 para não distorcer/destruir excessivamente as imagens 32x32
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

print(" PREPARANDO DADOS COM RAND-AUGMENT")

print("Carregando o dataset de treino com as novas transformações...")
train_data_aug = torchvision.datasets.SVHN(root='./data', split='train',
  download=True, transform=transform_augment)

train_loader_aug = torch.utils.data.DataLoader(train_data_aug, batch_size=64,
  shuffle=True, num_workers=2)

# 3. Criação de nova rede para treinar com os dados mais brutos
# (Assumindo que a variável 'device' e a classe 'DeepCNN' já estão na memória)
print(f"Dispositivo de Treinamento ativo: >>> {device.type.upper()} <<<")

model_aug = DeepCNN().to(device)
optimizer_aug = optim.Adam(model_aug.parameters(), lr=0.001)

best_acc_aug = 0.0
caminho_aug = 'modelo_deepcnn_randaugment.pth'
epochs_aug = 15

print(f"\nIniciando treinamento da DeepCNN com RandAugment por {epochs_aug} épocas...")

for epoch in range(epochs_aug):
    model_aug.train()
    running_loss = 0.0

    for i, data in enumerate(train_loader_aug, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        optimizer_aug.zero_grad()
        outputs = model_aug(inputs)
        loss = criterion(outputs, labels) # Pressupõe que criterion (CrossEntropyLoss) já foi instanciado
        loss.backward()
        optimizer_aug.step()

        running_loss += loss.item()

    loss_epoca = running_loss / len(train_loader_aug)

    model_aug.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = model_aug(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = (correct / total) * 100

    print(f"ÉPOCA [{epoch+1}/{epochs_aug}] | Perda (Loss): {loss_epoca:.4f} | Acurácia: {epoch_acc:.2f}%")

    if epoch_acc > best_acc_aug:
        best_acc_aug = epoch_acc
        torch.save(model_aug.state_dict(), caminho_aug)
        print(f"   ↪🚩Checkpoint guardado ({best_acc_aug:.2f}%)")

print(f"Treino com RandAugment Concluído ---> melhor modelo salvo em: '{caminho_aug}'")

#PARTE 5: usando dados de treinamentos extras

In [ ]:
print("CARREGANDO DADOS NÃO ROTULADOS (EXTRA SVHN)")

# 1. Definição das transformações fracas e fortes
transform_weak = transforms.Compose([
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_strong = transforms.Compose([
    transforms.RandAugment(num_ops=2, magnitude=10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 2. Classe customizada para retornar as duas versões da imagem
class UnlabeledDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, _ = self.dataset[idx]
        weak_img = transform_weak(image)
        strong_img = transform_strong(image)
        return weak_img, strong_img

print("Baixando/Carregando dados extras do SVHN (Aprox. 1.9 GB)...")
# Download das imagens extras
extra_data_raw = torchvision.datasets.SVHN(root='./data', split='extra', download=True)

# Sorteio de 50k imagens
print("Sorteando 50.000 imagens aleatórias para o FixMatch...")
indices = np.random.choice(len(extra_data_raw), 50000, replace=False)
extra_subset = Subset(extra_data_raw, indices)

unlabeled_dataset = UnlabeledDataset(extra_subset)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True, num_workers=2)

print("CONFIGURAÇÃO E TREINO: FIXMATCH")

# 3. Configuração do Treino do FixMatch
# Assumindo que model_aug (DeepCNN) já está carregado e treinado da etapa anterior
optimizer_fm = optim.Adam(model_aug.parameters(), lr=0.0005)

epochs_fm = 10
threshold = 0.97 # O modelo só confia no pseudo-rótulo se tiver > 97% de certeza

best_acc_fm = 0.0
caminho_fm = 'melhor_modelo_fixmatch50k.pth'

print(f"Iniciando treino Semi-Supervisionado por {epochs_fm} épocas (Threshold: {threshold*100}%)...")

for epoch in range(epochs_fm):
    model_aug.train()
    running_loss = 0.0
    batches_processed = 0

    for i, (inputs_weak, inputs_strong) in enumerate(unlabeled_loader):
        inputs_weak = inputs_weak.to(device)
        inputs_strong = inputs_strong.to(device)

        optimizer_fm.zero_grad()

        with torch.no_grad():
            outputs_weak = model_aug(inputs_weak)
            probs = F.softmax(outputs_weak, dim=1)
            max_probs, pseudo_labels = torch.max(probs, dim=1)

        mask = max_probs.ge(threshold).float()

        if mask.sum() > 0:
            outputs_strong = model_aug(inputs_strong)

            loss = (F.cross_entropy(outputs_strong, pseudo_labels, reduction='none') * mask).mean()

            loss.backward()
            optimizer_fm.step()
            running_loss += loss.item()
            batches_processed += 1

    loss_epoca = running_loss / max(1, batches_processed)

    model_aug.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = model_aug(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = (correct / total) * 100

    # Output embelezado da época
    print(f"ÉPOCA [{epoch+1}/{epochs_fm}] | Perda (Loss): {loss_epoca:.4f} | Acurácia: {epoch_acc:.2f}%")

    if epoch_acc > best_acc_fm:
        best_acc_fm = epoch_acc
        torch.save(model_aug.state_dict(), caminho_fm)
        print(f"   ↪🚩 Checkpoint guardado({best_acc_fm:.2f}%)")

print(f"Treino FixMatch Concluído ---> melhor modelo absoluto atingiu {best_acc_fm:.2f}%")
print(f"arquivo final salvo : '{caminho_fm}'")

#PARTE 6: AUMENTANDO A QUANT. DE IMAGENS DE TESTE

In [ ]:
print("CARREGANDO DADOS NÃO ROTULADOS (EXTRA SVHN)")

transform_weak = transforms.Compose([
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

transform_strong = transforms.Compose([
    transforms.RandAugment(num_ops=2, magnitude=10),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

class UnlabeledDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        image, _ = self.dataset[idx]
        weak_img = transform_weak(image)
        strong_img = transform_strong(image)
        return weak_img, strong_img

print("Baixando/Carregando dados extras do SVHN (Aprox. 1.9 GB)...")
extra_data_raw = torchvision.datasets.SVHN(root='./data', split='extra', download=True)

print("A selecionar 100.000 imagens aleatórias para o FixMatch Escalonado...")
indices = np.random.choice(len(extra_data_raw), 100000, replace=False) # Aumentado para 100k
extra_subset = Subset(extra_data_raw, indices)

unlabeled_dataset = UnlabeledDataset(extra_subset)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True, num_workers=2)

print("CONFIGURAÇÃO E TREINO ---> FIXMATCH (AJUSTE FINO)")

# 2. Configurar o Treino do FixMatch (Ajuste Fino)
# Reduzimos o Learning Rate para 0.0001 para não "destruir" os pesos bons
optimizer_fm = optim.Adam(model_aug.parameters(), lr=0.0001)

epochs_fm = 10 # Menos épocas, apenas para ajuste fino
threshold = 0.98 # Aumentamos o rigor: Só confia se tiver > 98% de certeza

best_acc_fm = 0.0
caminho_fm = 'melhor_modelo_fixmatch100k.pth'

print(f"\n Iniciando o treino Semi-Supervisionado Escalonado (FixMatch) com threshold de {threshold*100:.0f}%...")

model_aug.train()

for epoch in range(epochs_fm):
    running_loss = 0.0
    batches_processed = 0

    for i, (inputs_weak, inputs_strong) in enumerate(unlabeled_loader):
        inputs_weak = inputs_weak.to(device)
        inputs_strong = inputs_strong.to(device)

        optimizer_fm.zero_grad()

        with torch.no_grad():
            outputs_weak = model_aug(inputs_weak)
            probs = F.softmax(outputs_weak, dim=1)
            max_probs, pseudo_labels = torch.max(probs, dim=1)

        ## Máscara de Confiança muito mais severa (> 98%)
        mask = max_probs.ge(threshold).float()

        if mask.sum() > 0:
            outputs_strong = model_aug(inputs_strong)
            loss = (F.cross_entropy(outputs_strong, pseudo_labels, reduction='none') * mask).mean()

            loss.backward()
            optimizer_fm.step()
            running_loss += loss.item()
            batches_processed += 1

    loss_epoca = running_loss / max(1, batches_processed)

    model_aug.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = model_aug(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total

    print(f"ÉPOCA FixMatch: [{epoch+1}/{epochs_fm}] | Perda(LOSS) nas imagens extras: {loss_epoca:.4f} | Acurácia: {epoch_acc:.2f}%")

    if epoch_acc > best_acc_fm:
        best_acc_fm = epoch_acc
        torch.save(model_aug.state_dict(), caminho_fm)
        print(f"   ↪🚩 Checkpoint guardado ({best_acc_fm:.2f}%)")

    model_aug.train()

print(f"EXATIDÃO (ACCURACY) MÁXIMA FINAL APÓS FIXMATCH (100k): {best_acc_fm:.2f}%")
print(f"O arquivo final está salvo como: '{caminho_fm}'")

In [ ]:
print("A selecionar 150.000 imagens aleatórias para o FixMatch Escalonado...")
indices = np.random.choice(len(extra_data_raw), 150000, replace=False) # Aumentado para 150k
extra_subset = Subset(extra_data_raw, indices)

unlabeled_dataset = UnlabeledDataset(extra_subset)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True, num_workers=2)

print("CONFIGURAÇÃO E TREINO: FIXMATCH (AJUSTE FINO - 150k)")

# 2. Configurar o Treino do FixMatch (Ajuste Fino)
# Reduzimos o Learning Rate para 0.0001 para não "destruir" os pesos bons
optimizer_fm = optim.Adam(model_aug.parameters(), lr=0.0001)

epochs_fm = 10 # Menos épocas, apenas para ajuste fino
threshold = 0.96 # Aumentamos o rigor: Só confia se tiver > 96% de certeza

best_acc_fm = 0.0
caminho_fm = 'melhor_modelo_fixmatch_150k.pth'

print(f"\nIniciando o treino Semi-Supervisionado Escalonado (FixMatch) com threshold de {threshold*100:.0f}% e 150k de imagens...")

for epoch in range(epochs_fm):
    model_aug.train()
    running_loss = 0.0
    batches_processed = 0

    for i, (inputs_weak, inputs_strong) in enumerate(unlabeled_loader):
        inputs_weak = inputs_weak.to(device)
        inputs_strong = inputs_strong.to(device)

        optimizer_fm.zero_grad()

        with torch.no_grad():
            outputs_weak = model_aug(inputs_weak)
            probs = F.softmax(outputs_weak, dim=1)
            max_probs, pseudo_labels = torch.max(probs, dim=1)

        mask = max_probs.ge(threshold).float()

        if mask.sum() > 0:
            outputs_strong = model_aug(inputs_strong)
            loss = (F.cross_entropy(outputs_strong, pseudo_labels, reduction='none') * mask).mean()

            loss.backward()
            optimizer_fm.step()
            running_loss += loss.item()
            batches_processed += 1

    loss_epoca = running_loss / max(1, batches_processed)

    model_aug.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = model_aug(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total

    print(f"ÉPOCA FixMatch: [{epoch+1}/{epochs_fm}] | Perda(LOSS): {loss_epoca:.4f} | Acurácia: {epoch_acc:.2f}%")

    if epoch_acc > best_acc_fm:
        best_acc_fm = epoch_acc
        torch.save(model_aug.state_dict(), caminho_fm)
        print(f"   ↪🚩Checkpoint guardado ({best_acc_fm:.2f}%)")

print(f"EXATIDÃO (ACCURACY) MÁXIMA FINAL APÓS FIXMATCH (150k): {best_acc_fm:.2f}%")
print(f"arquivo final salvo: '{caminho_fm}'")

In [ ]:
print("A selecionar 200.000 imagens aleatórias para o FixMatch Escalonado...")
indices = np.random.choice(len(extra_data_raw), 200000, replace=False) # Aumentado para 200k
extra_subset = Subset(extra_data_raw, indices)

unlabeled_dataset = UnlabeledDataset(extra_subset)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True, num_workers=2)

print("CONFIGURAÇÃO E TREINO: FIXMATCH (AJUSTE FINO - 200k)")

# 2. Configurar o Treino do FixMatch (Ajuste Fino)
# Reduzimos o Learning Rate para 0.0001 para não "destruir" os pesos bons
optimizer_fm = optim.Adam(model_aug.parameters(), lr=0.0001)

epochs_fm = 10 # Menos épocas, apenas para ajuste fino
threshold = 0.96 # Aumentamos o rigor: Só confia se tiver > 96% de certeza

best_acc_fm = 0.0
caminho_fm = 'melhor_modelo_fixmatch_200k.pth'

print(f"\n Iniciando o treino Semi-Supervisionado Escalonado (FixMatch) com threshold de {threshold*100:.0f}% e 200k de imagens...")

for epoch in range(epochs_fm):
    model_aug.train()
    running_loss = 0.0
    batches_processed = 0

    for i, (inputs_weak, inputs_strong) in enumerate(unlabeled_loader):
        inputs_weak = inputs_weak.to(device)
        inputs_strong = inputs_strong.to(device)

        optimizer_fm.zero_grad()

        with torch.no_grad():
            outputs_weak = model_aug(inputs_weak)
            probs = F.softmax(outputs_weak, dim=1)
            max_probs, pseudo_labels = torch.max(probs, dim=1)

        mask = max_probs.ge(threshold).float()

        if mask.sum() > 0:
            outputs_strong = model_aug(inputs_strong)
            loss = (F.cross_entropy(outputs_strong, pseudo_labels, reduction='none') * mask).mean()

            loss.backward()
            optimizer_fm.step()
            running_loss += loss.item()
            batches_processed += 1

    loss_epoca = running_loss / max(1, batches_processed)

    model_aug.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = model_aug(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total

    print(f"ÉPOCA FixMatch: [{epoch+1}/{epochs_fm}] | Perda(LOSS): {loss_epoca:.4f} | Acurácia: {epoch_acc:.2f}%")

    if epoch_acc > best_acc_fm:
        best_acc_fm = epoch_acc
        torch.save(model_aug.state_dict(), caminho_fm)
        print(f"   ↪🚩Checkpoint guardado ({best_acc_fm:.2f}%)")

print(f"EXATIDÃO (ACCURACY) MÁXIMA FINAL APÓS FIXMATCH (200k): {best_acc_fm:.2f}%")
print(f"arquivo final salvo: '{caminho_fm}'")

In [ ]:
print("A selecionar 300.000 imagens aleatórias para o FixMatch Escalonado...")
indices = np.random.choice(len(extra_data_raw), 300000, replace=False) # Aumentado para 300k
extra_subset = Subset(extra_data_raw, indices)

unlabeled_dataset = UnlabeledDataset(extra_subset)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True, num_workers=2)

print("CONFIGURAÇÃO E TREINO: FIXMATCH (AJUSTE FINO - 300k)")

# 2. Configurar o Treino do FixMatch (Ajuste Fino)
# Reduzimos o Learning Rate para 0.0001 para não "destruir" os pesos bons
optimizer_fm = optim.Adam(model_aug.parameters(), lr=0.0001)

epochs_fm = 10 # Menos épocas, apenas para ajuste fino
threshold = 0.95 # Aumentamos o rigor: Só confia se tiver > 95% de certeza

best_acc_fm = 0.0
caminho_fm = 'melhor_modelo_fixmatch_300k.pth'

print(f"\n Iniciando o treino Semi-Supervisionado Escalonado (FixMatch) com threshold de {threshold*100:.0f}% e 300k de imagens...")

for epoch in range(epochs_fm):
    model_aug.train()
    running_loss = 0.0
    batches_processed = 0

    for i, (inputs_weak, inputs_strong) in enumerate(unlabeled_loader):
        inputs_weak = inputs_weak.to(device)
        inputs_strong = inputs_strong.to(device)

        optimizer_fm.zero_grad()

        with torch.no_grad():
            outputs_weak = model_aug(inputs_weak)
            probs = F.softmax(outputs_weak, dim=1)
            max_probs, pseudo_labels = torch.max(probs, dim=1)

        mask = max_probs.ge(threshold).float()

        if mask.sum() > 0:
            outputs_strong = model_aug(inputs_strong)
            loss = (F.cross_entropy(outputs_strong, pseudo_labels, reduction='none') * mask).mean()

            loss.backward()
            optimizer_fm.step()
            running_loss += loss.item()
            batches_processed += 1

    loss_epoca = running_loss / max(1, batches_processed)

    model_aug.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = model_aug(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total

    print(f"ÉPOCA FixMatch: [{epoch+1}/{epochs_fm}] | Perda(LOSS): {loss_epoca:.4f} | Acurácia: {epoch_acc:.2f}%")

    if epoch_acc > best_acc_fm:
        best_acc_fm = epoch_acc
        torch.save(model_aug.state_dict(), caminho_fm)
        print(f"   ↪🚩Checkpoint guardado ({best_acc_fm:.2f}%)")

print(f"EXATIDÃO (ACCURACY) MÁXIMA FINAL APÓS FIXMATCH (300k): {best_acc_fm:.2f}%")
print(f"arquivo final salvo: '{caminho_fm}'")

In [ ]:
print("A selecionar 400.000 imagens aleatórias para o FixMatch Escalonado...")
indices = np.random.choice(len(extra_data_raw), 400000, replace=False) # Aumentado para 400k
extra_subset = Subset(extra_data_raw, indices)

unlabeled_dataset = UnlabeledDataset(extra_subset)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True, num_workers=2)

print("CONFIGURAÇÃO E TREINO: FIXMATCH (AJUSTE FINO - 400k)")

# 2. Configurar o Treino do FixMatch (Ajuste Fino)
# Reduzimos o Learning Rate para 0.0001 para não "destruir" os pesos bons
optimizer_fm = optim.Adam(model_aug.parameters(), lr=0.0001)

epochs_fm = 10 # Menos épocas, apenas para ajuste fino
threshold = 0.95 # Aumentamos o rigor: Só confia se tiver > 95% de certeza

best_acc_fm = 0.0
caminho_fm = 'melhor_modelo_fixmatch_400k.pth'

print(f"\n Iniciando o treino Semi-Supervisionado Escalonado (FixMatch) com threshold de {threshold*100:.0f}% e 400k de imagens...")

for epoch in range(epochs_fm):
    model_aug.train()
    running_loss = 0.0
    batches_processed = 0

    for i, (inputs_weak, inputs_strong) in enumerate(unlabeled_loader):
        inputs_weak = inputs_weak.to(device)
        inputs_strong = inputs_strong.to(device)

        optimizer_fm.zero_grad()

        with torch.no_grad():
            outputs_weak = model_aug(inputs_weak)
            probs = F.softmax(outputs_weak, dim=1)
            max_probs, pseudo_labels = torch.max(probs, dim=1)

        mask = max_probs.ge(threshold).float()

        if mask.sum() > 0:
            outputs_strong = model_aug(inputs_strong)
            loss = (F.cross_entropy(outputs_strong, pseudo_labels, reduction='none') * mask).mean()

            loss.backward()
            optimizer_fm.step()
            running_loss += loss.item()
            batches_processed += 1

    loss_epoca = running_loss / max(1, batches_processed)

    model_aug.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = model_aug(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total

    print(f"ÉPOCA FixMatch: [{epoch+1}/{epochs_fm}] | Perda(LOSS): {loss_epoca:.4f} | Acurácia: {epoch_acc:.2f}%")

    if epoch_acc > best_acc_fm:
        best_acc_fm = epoch_acc
        torch.save(model_aug.state_dict(), caminho_fm)
        print(f"   ↪🚩Checkpoint guardado ({best_acc_fm:.2f}%)")

print(f"EXATIDÃO (ACCURACY) MÁXIMA FINAL APÓS FIXMATCH (400k): {best_acc_fm:.2f}%")
print(f"arquivo final salvo: '{caminho_fm}'")

In [ ]:
print("A selecionar todas as imagens extras para o FixMatch Escalonado...")
indices = np.random.choice(len(extra_data_raw), 531131, replace=False) # Aumentado para todas as imagens
extra_subset = Subset(extra_data_raw, indices)

unlabeled_dataset = UnlabeledDataset(extra_subset)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=64, shuffle=True, num_workers=2)

print("CONFIGURAÇÃO E TREINO: FIXMATCH (AJUSTE FINO - 531k)")

# 2. Configurar o Treino do FixMatch (Ajuste Fino)
# Reduzimos o Learning Rate para 0.0001 para não "destruir" os pesos bons
optimizer_fm = optim.Adam(model_aug.parameters(), lr=0.0001)

epochs_fm = 10 # Menos épocas, apenas para ajuste fino
threshold = 0.95 # Aumentamos o rigor: Só confia se tiver > 95% de certeza

best_acc_fm = 0.0
caminho_fm = 'melhor_modelo_fixmatch_full531k.pth'

print(f"\n Iniciando o treino Semi-Supervisionado Escalonado (FixMatch) com threshold de {threshold*100:.0f}% e 531k de imagens...")

for epoch in range(epochs_fm):
    model_aug.train()
    running_loss = 0.0
    batches_processed = 0

    for i, (inputs_weak, inputs_strong) in enumerate(unlabeled_loader):
        inputs_weak = inputs_weak.to(device)
        inputs_strong = inputs_strong.to(device)

        optimizer_fm.zero_grad()

        with torch.no_grad():
            outputs_weak = model_aug(inputs_weak)
            probs = F.softmax(outputs_weak, dim=1)
            max_probs, pseudo_labels = torch.max(probs, dim=1)

        mask = max_probs.ge(threshold).float()

        if mask.sum() > 0:
            outputs_strong = model_aug(inputs_strong)
            loss = (F.cross_entropy(outputs_strong, pseudo_labels, reduction='none') * mask).mean()

            loss.backward()
            optimizer_fm.step()
            running_loss += loss.item()
            batches_processed += 1

    loss_epoca = running_loss / max(1, batches_processed)

    model_aug.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data in test_loader:
            inputs, labels = data[0].to(device), data[1].to(device)
            outputs = model_aug(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_acc = 100 * correct / total

    print(f"ÉPOCA FixMatch: [{epoch+1}/{epochs_fm}] | Perda(LOSS): {loss_epoca:.4f} | Acurácia: {epoch_acc:.2f}%")

    if epoch_acc > best_acc_fm:
        best_acc_fm = epoch_acc
        torch.save(model_aug.state_dict(), caminho_fm)
        print(f"   ↪🚩Checkpoint guardado ({best_acc_fm:.2f}%)")

print(f"EXATIDÃO (ACCURACY) MÁXIMA FINAL APÓS FIXMATCH (531k): {best_acc_fm:.2f}%")
print(f"arquivo final salvo: '{caminho_fm}'")

PARTE 7 - AVALIAÇÃO DE EVOLUÇÃO E MÉTRICAS


In [ ]:
print("EXTRAÇÃO DE PREDIÇÕES DO DATASET DE TESTE")

# 1. Coletando predições no Dataset de Teste
model_aug.eval()
all_preds = []
all_labels = []

print("Carregando todas as imagens de teste para cálculo de métricas...")

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)

        # Cálculo da predição em probabilidade
        outputs = model_aug(inputs)

        # Conversão das probabilidades da classe vencedora
        preds = torch.argmax(outputs, dim=1)

        # Armazenamento dos rótulos reais e das predições --> envia para a CPU
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

y_true = np.array(all_labels)
y_pred = np.array(all_preds)

# Definição dos nomes das classes ---> Dígitos de 0 a 9
class_names = [f"Dígito {i}" for i in range(10)]

print("Predições finalizadas")

print("RELATÓRIO DE CLASSIFICAÇÃO POR DÍGITO")

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

print("\nGerando Matriz de Confusão...")

# Geração dos dados da matriz de confusão usando Scikit-Learn
cm = confusion_matrix(y_true, y_pred)

# Configuração visual avançada com Seaborn
plt.figure(figsize=(12, 9))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
      xticklabels=class_names, yticklabels=class_names,
      cbar_kws={'label': 'Frequência'})

plt.title('Matriz de Confusão: Classificação de Dígitos SVHN (FixMatch)', fontsize=16, fontweight='bold', pad=20)
plt.ylabel('Dígito Real (Gabarito)', fontsize=12, fontweight='bold')
plt.xlabel('Dígito Predito pelo Modelo', fontsize=12, fontweight='bold')

plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()

plt.show()